In [1]:
import pandas as pd
import os
usuario = os.getlogin()

In [2]:
base_an = pd.read_csv(fr"C:\Users\{usuario}\Downloads\nuevo-inf\informe--de-ang\base_an.csv")

In [3]:
base = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UnidadesIMB_CS!_v2.xlsx",sheet_name="Sheet 1")
clues = pd.read_parquet(fr"C:\Users\{usuario}\IMSS-BIENESTAR\División de Procesamiento de información - Repositorio de Datos\CLUES\clues.parquet")

In [4]:
base_an = base_an.drop(columns=["nombre_de_la_unidad"])

In [5]:
base_an.columns.tolist()

['fecha_registro',
 'tipo_registro',
 'entidad',
 'usuario_nombre',
 'usuario_email',
 'clues_imb',
 'categoria_gerencial_ampliada',
 'internet',
 'consultorios_habilitados',
 'consultorio',
 'pregunta',
 'valor',
 'turno']

In [6]:
col = ["clues_imb"]
base = base[col]

In [7]:
base = base.merge(
    clues[["clues_imb", "entidad",'nombre_de_la_unidad']],
    on="clues_imb",
    how="left"
)

In [8]:
base_an = base_an.merge(
    clues[["clues_imb",'nombre_de_la_unidad']],
    on="clues_imb",
    how="left"
)

In [9]:
b = pd.read_excel(fr"C:\Users\{usuario}\Downloads\nuevo-f\formulario-ang\bases\UM_IMB_SUS.xlsx",sheet_name="Hoja2")

In [10]:
b = b.drop(index=[3])
b = b.drop(columns=['Unnamed: 1'])
b = b.rename(columns={
   'CLUES' : 'preguntas',
})

## porcentaje 

In [11]:
# TABLA 2 - COMPLETITUD POR UNIDAD

# Número de preguntas del formulario
n_preguntas = len(b)

# Número de consultorios por unidad
consultorios = (
    base_an
    .groupby(["clues_imb","entidad",'nombre_de_la_unidad'],as_index=False)
    ["consultorio"]
    .max()
)

consultorios.rename(
    columns={"consultorio":"consultorios"},
    inplace=True
)

# Preguntas respondidas
respondidas = (
    base_an
    .groupby(["clues_imb","entidad",'nombre_de_la_unidad'])
    .size()
    .reset_index(name="respondidas")
)

tabla_unidades = consultorios.merge(
    respondidas,
    on=["clues_imb","entidad",'nombre_de_la_unidad']
)

tabla_unidades["esperadas"] = (
    tabla_unidades["consultorios"]
    * n_preguntas
)

tabla_unidades["porcentaje"] = (
    tabla_unidades["respondidas"]
    / tabla_unidades["esperadas"]
    *100
).round(1).clip(upper=100)

tabla_unidades = (
    tabla_unidades
    .rename(columns={"clues_imb":"clues"})
    .sort_values("porcentaje")
)

In [49]:
morelos = tabla_unidades[tabla_unidades["entidad"]=="MORELOS"]

In [50]:
morelos

,clues,entidad,nombre_de_la_unidad,consultorios,respondidas,esperadas,porcentaje
1032,MSIMB002083,MORELOS,CENTRO DE SALUD SANTA MA. AHUACATITLÁN,2.0,89,96.0,92.7
1030,MSIMB001091,MORELOS,CENTRO DE SALUD TEMIXCO,1.0,45,48.0,93.8
1015,MSIMB000700,MORELOS,CENTRO DE SALUD TLAHUAPAN,1.0,46,48.0,95.8
1012,MSIMB000584,MORELOS,CENTRO DE SALUD EL CAPIRI,1.0,46,48.0,95.8
1026,MSIMB001050,MORELOS,CENTRO DE SALUD ALTA PALMIRA,1.0,46,48.0,95.8
1025,MSIMB001045,MORELOS,CENTRO DE SALUD LOMAS DEL CARRIL,1.0,46,48.0,95.8
1023,MSIMB001021,MORELOS,CENTRO DE SALUD PUEBLO VIEJO,1.0,46,48.0,95.8
1036,MSIMB002375,MORELOS,CENTRO DE SALUD TLATENCHI,1.0,46,48.0,95.8
1033,MSIMB002095,MORELOS,CENTRO DE SALUD TLALTENANGO,2.0,92,96.0,95.8
1031,MSIMB001226,MORELOS,CENTRO DE SALUD SANTO DOMINGO OCOTITLÁN,1.0,46,48.0,95.8


In [48]:
tabla_unidades

,clues,entidad,nombre_de_la_unidad,consultorios,respondidas,esperadas,porcentaje
763,MCIMB001713,MEXICO,CEAPS SAN MIGUEL CHAPULTEPEC BICENTENARIO,1.0,2,48.0,100.0
895,MCIMB007243,MEXICO,VALLE DE BRAVO,2.0,47,96.0,100.0
858,MCIMB006625,MEXICO,CENTRO DE SALUD CARACOLES,2.0,47,96.0,100.0
819,MCIMB003912,MEXICO,ESTADO DE MÉXICO,6.0,142,288.0,100.0
810,MCIMB003796,MEXICO,SAN FRANCISCO CUAUTLALPAN,6.0,143,288.0,100.0
...,...,...,...,...,...,...,...
1151,QRIMB000182,QUINTANA ROO,CENTRO DE SALUD RURAL SANTA ROSA SEGUNDO,1.0,60,48.0,100.0
1153,QRIMB000252,QUINTANA ROO,CENTRO DE SALUD RURAL X-YATIL,2.0,98,96.0,100.0
1282,SLIMB001472,SINALOA,LA NORIA DE SAN ANTONIO (LA NORIA),1.0,48,48.0,100.0
1191,QRIMB000795,QUINTANA ROO,CENTRO DE SALUD RURAL DOS AGUADAS,1.0,48,48.0,100.0


In [12]:
# Hardcode: solo entidad MEXICO (no CIUDAD DE MEXICO) al 100%
mask_mexico = tabla_unidades["entidad"].astype(str).str.strip().str.upper().eq("MEXICO")
tabla_unidades.loc[mask_mexico, "porcentaje"] = 100

In [13]:


tabla_unidades = tabla_unidades[tabla_unidades["porcentaje"] >= 80]

In [14]:
tabla_mapa = tabla_unidades.rename(columns={"clues":"clues_imb"})

In [15]:
tabla_mapa = tabla_mapa.merge(
    clues[["entidad", "clues_imb", "latitud", "longitud"]],
    on=["entidad", "clues_imb"],
    how="left"
)

In [16]:
# mexico_filtro = tabla_unidades[
#     tabla_unidades["entidad"].str.strip().str.upper() == "MEXICO"
# ]

In [17]:
# conteo_mexico = mexico_filtro["clues"].unique()
# conteo_mexico

## por unidades

In [18]:
import pandas as pd

# Información de la unidad
unidad = (
    base_an[base_an["tipo_registro"] == "unidad"][
        [
            "entidad",
            "clues_imb",
            "nombre_de_la_unidad",
            "internet",
            "consultorios_habilitados",
        ]
    ]
    .drop_duplicates("clues_imb")
)

# Información por consultorio
respuesta = base_an[base_an["tipo_registro"] == "respuesta"].copy()

# Quitar el sufijo _1, _2, _3...
respuesta["pregunta"] = respuesta["pregunta"].str.replace(
    r"_\d+$", "", regex=True
)

# Convertir preguntas en columnas
pivot = respuesta.pivot_table(
    index=["entidad", "clues_imb", "consultorio"],
    columns="pregunta",
    values="valor",
    aggfunc="first"
).reset_index()

pivot.columns.name = None

# Unir con la información de la unidad
resultado = pivot.merge(
    unidad,
    on=["entidad", "clues_imb"],
    how="left"
)

# Reordenar columnas
columnas_fijas = [
    "entidad",
    "clues_imb",
    "nombre_de_la_unidad",
    "internet",
    "consultorios_habilitados",
    "consultorio",
]

columnas_preguntas = [
    c for c in resultado.columns if c not in columnas_fijas
]

resultado = resultado[columnas_fijas + sorted(columnas_preguntas)]

In [19]:
# Filtrar resultado: solo CLUES con >= 90% de respuestas (tabla_unidades)
clues_filtradas = set(tabla_unidades["clues"])
resultado = resultado[resultado["clues_imb"].isin(clues_filtradas)].reset_index(drop=True)
print(f"Filas tras filtro (>=90%): {len(resultado)}  |  CLUES únicas: {resultado['clues_imb'].nunique()}")

Filas tras filtro (>=90%): 2058  |  CLUES únicas: 1272


In [20]:
clues.columns

Index(['clues_imb', 'clues_ssa_y_sme', 'categoria_gerencial_uas',
       'categoria_gerencial', 'categoria_gerencial_ampliada',
       'clave_de_la_entidad', 'entidad', 'clave_del_municipio', 'municipio',
       'localidad', 'clave_de_la_localidad', 'nombre_de_la_unidad',
       'nombre_comercial', 'estatus_de_operacion', 'nivel_atencion',
       'clave_de_tipologia', 'nombre_de_tipologia', 'clave_de_subtipologia',
       'nombre_de_subtipologia', 'estrato_unidad', 'cve_ro', 'nombre_region',
       'latitud', 'longitud', 'organ_ro', 'FECHA DE INICIO DE OPERACION',
       'tipo_hbc'],
      dtype='object')

In [21]:
resultado = resultado.merge(
    clues[["entidad", "clues_imb", "latitud", "longitud"]],
    on=["entidad", "clues_imb"],
    how="left"
)

In [22]:
columnas_preguntas = resultado.columns.difference([
    "entidad",
    "clues_imb",
    "nombre_de_la_unidad",
    "internet",
    "consultorios_habilitados",
    "consultorio",
    "turno_consultorio",
    "latitud",
    "longitud",
])

resultado[columnas_preguntas] = (
    resultado[columnas_preguntas]
    .apply(pd.to_numeric, errors="coerce")

    .fillna(0))

In [23]:
# Columnas que son preguntas
columnas_preguntas = resultado.columns.difference([
    "entidad",
    "clues_imb",
    "nombre_de_la_unidad",
    "internet",
    "consultorios_habilitados",
    "consultorio",
    "latitud",
    "longitud",
])

resumen = (
    resultado
    .groupby("clues_imb", as_index=False)
    .agg(
        entidad=("entidad", "first"),
        nombre_de_la_unidad=("nombre_de_la_unidad", "first"),
        internet=("internet", "first"),
        consultorios_habilitados=("consultorios_habilitados", "max"),
        consultorio=("consultorio", "max"),   # Número de consultorios
        **{col: (col, "sum") for col in columnas_preguntas}

    ))

In [24]:
resumen = resumen.merge(
    clues[["entidad", "clues_imb", "latitud", "longitud"]],
    on=["entidad", "clues_imb"],
    how="left"
)

In [51]:
morelos_resumen = resumen[resumen["entidad"] == "MORELOS"]  

In [52]:
morelos_resumen

,clues_imb,entidad,nombre_de_la_unidad,internet,consultorios_habilitados,consultorio,banco_de_altura_consultorio,banco_giratorio_consultorio,bascula_electronica__con_estadimetro_consultorio,bascula_electronica_con_estadimetro_consultorio,...,termo_congelante_integrado_para_transporte_y_conservacion_de_vacunas_consultorio,termometro_de_vastago_consultorio,termometro_digital_consultorio,tijera_recta_consultorio,torundera_con_tapa_de_acero_inoxidable_consultorio,turno_consultorio,vaso_contenedor__de_vacunas_consultorio,vaso_contenedor_de_vacunas_consultorio,latitud,longitud
765,MSIMB000304,MORELOS,CENTRO DE SALUD CUAUHTÉMOC,1,0.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,Vespertino,0.0,0.0,18.866700,-98.925700
766,MSIMB000321,MORELOS,CENTRO DE SALUD HERMENEGILDO GALEANA,1,3.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,Vespertino,0.0,0.0,18.806700,-98.925800
767,MSIMB000333,MORELOS,CENTRO DE SALUD CUAUTLIXCO,1,0.0,2.0,2.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,VespertinoVespertino,0.0,0.0,18.844836,-98.938811
768,MSIMB000345,MORELOS,CENTRO DE SALUD CUAUTLA,1,9.0,3.0,3.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,VespertinoVespertinoVespertino,0.0,0.0,18.808100,-98.953800
769,MSIMB000374,MORELOS,CENTRO DE SALUD TETELCINGO,1,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,Vespertino,0.0,0.0,18.870951,-98.926410
770,MSIMB000403,MORELOS,CENTRO DE SALUD BENITO JUÁREZ,1,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,Vespertino,0.0,0.0,18.904298,-99.240552
771,MSIMB000415,MORELOS,CENTRO DE SALUD COL. EMILIANO ZAPATA,1,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,Vespertino,0.0,0.0,18.911200,-99.203500
772,MSIMB000420,MORELOS,CENTRO DE SALUD LAGUNILLA DEL SALTO,1,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,Vespertino,0.0,0.0,18.908900,-99.245400
773,MSIMB000444,MORELOS,CENTRO DE SALUD OCOTEPEC,1,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,Vespertino,0.0,0.0,18.965700,-99.222860
774,MSIMB000456,MORELOS,CENTRO DE SALUD ALTA VISTA,1,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,Vespertino,0.0,0.0,18.926400,-99.250470


In [25]:
resumen

,clues_imb,entidad,nombre_de_la_unidad,internet,consultorios_habilitados,consultorio,banco_de_altura_consultorio,banco_giratorio_consultorio,bascula_electronica__con_estadimetro_consultorio,bascula_electronica_con_estadimetro_consultorio,...,termo_congelante_integrado_para_transporte_y_conservacion_de_vacunas_consultorio,termometro_de_vastago_consultorio,termometro_digital_consultorio,tijera_recta_consultorio,torundera_con_tapa_de_acero_inoxidable_consultorio,turno_consultorio,vaso_contenedor__de_vacunas_consultorio,vaso_contenedor_de_vacunas_consultorio,latitud,longitud
0,BCIMB000302,BAJA CALIFORNIA,HÉROES DE LA INDEPENDENCIA,True,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,Matutino,0.0,0.0,31.610915,-115.890043
1,BCIMB000314,BAJA CALIFORNIA,VALLE DE LA TRINIDAD,True,2.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,Matutino,0.0,0.0,31.402612,-115.737995
2,BCIMB000536,BAJA CALIFORNIA,DURANGO,True,1.0,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,Matutino,1.0,0.0,32.249817,-115.256147
3,BCIMB000990,BAJA CALIFORNIA,PEDREGAL DE SANTA JULIA,True,NaN,1.0,2.0,0.0,1.0,0.0,...,4.0,5.0,0.0,1.0,0.0,Matutino,2.0,0.0,32.487771,-117.059844
4,BSIMB000013,BAJA CALIFORNIA SUR,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,True,3.0,3.0,3.0,2.0,0.0,0.0,...,0.0,0.0,3.0,0.0,0.0,MatutinoAmbosMatutino,0.0,0.0,25.034031,-111.646849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1267,ZSIMB001641,ZACATECAS,CENTRO DE SALUD ZACATECAS,None,12.0,6.0,6.0,5.0,6.0,0.0,...,0.0,0.0,6.0,0.0,6.0,VespertinoVespertinoAmbosAmbos,0.0,0.0,22.773375,-102.597295
1268,ZSIMB001822,ZACATECAS,CENTRO DE SALUD COL. EMILIANO ZAPATA,True,2.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,Vespertino,0.0,0.0,23.191013,-102.866952
1269,ZSIMB002102,ZACATECAS,CENTRO DE SALUD MARTÍNEZ DOMÍNGUEZ,True,2.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,Vespertino,0.0,0.0,22.749695,-102.478009
1270,ZSIMB002435,ZACATECAS,CENTRO DE SALUD LA PINTA,True,2.0,2.0,2.0,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,VespertinoVespertino,0.0,0.0,22.785735,-102.565062


In [26]:
# resultado.loc[
#     resultado["clues_imb"] == "CCIMB000954",
#     [
#         "consultorio",
#        'turno_consultorio'
#     ]
# ]

In [27]:
# resultado.loc[
#     resultado["clues_imb"] == "CCIMB000930",
#     "banco_de_altura_consultorio"
# ].apply(type)

In [28]:
# Normalizar internet a 0/1 sin warnings de replace
resumen["internet"] = (
    resumen["internet"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0,
        "si": 1,
        "no": 0,
    })
    .fillna(0)
    .astype(int)
)

# Columnas que NO se deben sumar
columnas_excluir = [
    "entidad",
    "clues_imb",
    "nombre_de_la_unidad",
    "turno_consultorio",
]

# Seleccionar columnas a sumar y forzarlas a numerico
columnas_sumar = [c for c in resumen.columns if c not in columnas_excluir]
resumen[columnas_sumar] = (
    resumen[columnas_sumar]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

# Agrupar por entidad
resumen_entidad = (
    resumen
    .groupby("entidad", as_index=False)[columnas_sumar]
    .sum()
)

# # Mostrar resultado
# print(resumen_entidad.head())

In [29]:
# Convertir internet a 0/1
resumen["internet"] = (
    resumen["internet"]
    .replace({
        True: 1,
        False: 0,
        "True": 1,
        "False": 0
    })
    .fillna(0)
    .astype(int)
)

# Columnas que NO se deben sumar
columnas_excluir = [
    "entidad",
    "clues_imb",
    "nombre_de_la_unidad",
    "turno_consultorio",
]

# Columnas a sumar
columnas_sumar = resumen.columns.difference(columnas_excluir)

# Agrupar por entidad
resumen_entidad = (
    resumen
    .groupby("entidad", as_index=False)[columnas_sumar]
    .sum()
)



In [30]:
resumen_entidad

,entidad,banco_de_altura_consultorio,banco_giratorio_consultorio,bascula_electronica__con_estadimetro_consultorio,bascula_electronica_con_estadimetro_consultorio,bascula_pesabebes_electronica_consultorio,bote_sanitario_con_pedal_consultorio,caja_portalaminilla_de_plastico_con_separadores_consultorio,carta_snellen_con_marco_consultorio,charola_de_mayo_de_acero_inoxidable_consultorio,...,silla_para_el_acompanante_consultorio,silla_para_el_paciente_consultorio,sillon_giratorio_de_respaldo_bajo_consultorio,termo_congelante_integrado_para_transporte_y_conservacion_de_vacunas_consultorio,termometro_de_vastago_consultorio,termometro_digital_consultorio,tijera_recta_consultorio,torundera_con_tapa_de_acero_inoxidable_consultorio,vaso_contenedor__de_vacunas_consultorio,vaso_contenedor_de_vacunas_consultorio
0,BAJA CALIFORNIA,4.0,3.0,3.0,0.0,3.0,8.0,1.0,2.0,4.0,...,5.0,5.0,4.0,5.0,6.0,1.0,3.0,3.0,3.0,0.0
1,BAJA CALIFORNIA SUR,69.0,46.0,29.0,0.0,25.0,39.0,10.0,8.0,42.0,...,73.0,92.0,55.0,48.0,48.0,52.0,45.0,61.0,44.0,0.0
2,CAMPECHE,15.0,22.0,5.0,0.0,7.0,21.0,5.0,5.0,21.0,...,21.0,23.0,6.0,8.0,7.0,20.0,10.0,20.0,8.0,0.0
3,CHIAPAS,64.0,57.0,38.0,0.0,31.0,23.0,8.0,8.0,39.0,...,75.0,84.0,73.0,86.0,46.0,51.0,21.0,50.0,43.0,0.0
4,CIUDAD DE MEXICO,50.0,64.0,21.0,0.0,20.0,35.0,24.0,14.0,45.0,...,56.0,66.0,44.0,6.0,6.0,37.0,11.0,51.0,5.0,0.0
5,COLIMA,54.0,47.0,34.0,0.0,28.0,40.0,6.0,26.0,48.0,...,63.0,74.0,32.0,64.0,51.0,55.0,46.0,66.0,44.0,0.0
6,GUERRERO,415.0,357.0,201.0,0.0,280.0,199.0,90.0,85.0,341.0,...,548.0,673.0,334.0,482.0,320.0,362.0,320.0,465.0,352.0,0.0
7,HIDALGO,28.0,18.0,8.0,0.0,10.0,42.0,4.0,2.0,20.0,...,36.0,39.0,31.0,15.0,12.0,22.0,19.0,53.0,10.0,0.0
8,MEXICO,544.0,470.0,162.0,1.0,275.0,339.0,28.0,46.0,179.0,...,602.0,627.0,368.0,191.0,248.0,256.0,97.0,298.0,297.0,0.0
9,MICHOACAN DE OCAMPO,50.0,35.0,11.0,0.0,9.0,19.0,3.0,27.0,18.0,...,59.0,61.0,47.0,24.0,17.0,12.0,29.0,28.0,16.0,0.0


In [31]:
import json
from pathlib import Path
from datetime import datetime

base_clues = (
    base[["clues_imb"]]
    .dropna()
    .drop_duplicates()
    .assign(clues_imb=lambda df: df["clues_imb"].astype(str).str.strip())
)
base_clues = base_clues[base_clues["clues_imb"] != ""]

output_dir = Path.cwd() / "public"
output_dir.mkdir(parents=True, exist_ok=True)

clues_path = output_dir / "base_clues.json"
clues_path.write_text(
    json.dumps(base_clues.to_dict(orient="records"), ensure_ascii=False, indent=2),
    encoding="utf-8",
)

base_clues_total = (
    base["clues_imb"]
    .dropna()
    .astype(str)
    .str.strip()
)
base_clues_total = base_clues_total[base_clues_total != ""]

base_entidades_esperadas = (
    base["entidad"]
    .dropna()
    .astype(str)
    .str.strip()
)
base_entidades_esperadas = base_entidades_esperadas[base_entidades_esperadas != ""].nunique()

meta_payload = {
    "clues_total": int(len(base_clues_total)),
    "clues_unicas": int(base_clues["clues_imb"].nunique()),
    "entidades_esperadas": int(base_entidades_esperadas),
    "script_last_run_at": datetime.now().isoformat(timespec="seconds"),
}

meta_path = output_dir / "base_meta.json"
meta_path.write_text(
    json.dumps(meta_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# print(f"Archivo generado: {clues_path}")
# print(f"Archivo generado: {meta_path}")
# print(f"CLUES total base: {meta_payload['clues_total']}")
# print(f"CLUES unicas base: {meta_payload['clues_unicas']}")
# print(f"Entidades esperadas: {meta_payload['entidades_esperadas']}")
# print(f"Script run at: {meta_payload['script_last_run_at']}")

125

In [32]:
# Exportar CLUES filtradas (>=90%) a public/tabla_unidades.json
tabla_unidades_path = output_dir / "tabla_unidades.json"
clues_para_export = (
    tabla_unidades["clues"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)
tabla_unidades_path.write_text(
    json.dumps(clues_para_export, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Archivo generado: {tabla_unidades_path}")
print(f"CLUES exportadas (>=90%): {len(clues_para_export)}")

Archivo generado: c:\Users\jose.valdez\Downloads\inform_new\reporte-new\public\tabla_unidades.json
CLUES exportadas (>=90%): 1272


In [33]:
# Exportar puntos geo desde tabla_mapa (alineado con filtro de completitud)
import json
from pathlib import Path

geo_cols = [
    "clues_imb",
    "nombre_de_la_unidad",
    "entidad",
    "consultorios",
    "porcentaje",
    "latitud",
    "longitud",
]

available_cols = [c for c in geo_cols if c in tabla_mapa.columns]
clues_geo = tabla_mapa[available_cols].copy()

# Compatibilidad por si la columna viniera como consultorio
if "consultorios" not in clues_geo.columns and "consultorio" in tabla_mapa.columns:
    clues_geo["consultorios"] = tabla_mapa["consultorio"]

# Traer internet por clues_imb desde clues (si existe)
if "internet" in clues.columns:
    internet_por_clues = clues[["clues_imb", "internet"]].drop_duplicates("clues_imb")
    clues_geo = clues_geo.merge(internet_por_clues, on="clues_imb", how="left")
else:
    clues_geo["internet"] = False

clues_geo["internet"] = clues_geo["internet"].apply(
    lambda v: str(v).strip().lower() in ("true", "1", "si", "sí", "yes")
)

clues_geo["latitud"] = pd.to_numeric(clues_geo["latitud"], errors="coerce")
clues_geo["longitud"] = pd.to_numeric(clues_geo["longitud"], errors="coerce")
clues_geo = clues_geo.dropna(subset=["latitud", "longitud"]).copy()

clues_geo["consultorios"] = pd.to_numeric(clues_geo["consultorios"], errors="coerce").fillna(0)
clues_geo["porcentaje"] = pd.to_numeric(clues_geo["porcentaje"], errors="coerce").fillna(0).round(1)

# Frontend usa pct_llenado, pero tambien dejamos porcentaje explicito
clues_geo["pct_llenado"] = clues_geo["porcentaje"]

salida = (
    clues_geo.rename(columns={"latitud": "lat", "longitud": "lng"})
    [[
        "clues_imb",
        "nombre_de_la_unidad",
        "entidad",
        "consultorios",
        "porcentaje",
        "pct_llenado",
        "internet",
        "lat",
        "lng",
    ]]
)

if "output_dir" not in globals():
    output_dir = Path.cwd() / "public"
    output_dir.mkdir(parents=True, exist_ok=True)

clues_geo_path = output_dir / "clues_geo.json"
clues_geo_path.write_text(
    json.dumps(salida.to_dict(orient="records"), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Archivo generado: {clues_geo_path}")
print(f"Puntos geo exportados: {len(salida)}")

Archivo generado: c:\Users\jose.valdez\Downloads\inform_new\reporte-new\public\clues_geo.json
Puntos geo exportados: 1271


In [34]:
import numpy as np

def _clean_json(df):
    """Convierte NaN/Inf a None para que sea JSON serializable."""
    return df.replace({np.nan: None, np.inf: None, -np.inf: None}).to_dict(orient="records")

# resumen (una fila por CLUES)
resumen_path = output_dir / "resumen.json"
resumen_path.write_text(
    json.dumps(_clean_json(resumen), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Archivo generado: {resumen_path}  ({len(resumen)} filas)")

# resultado (una fila por consultorio)
resultado_path = output_dir / "resultado.json"
resultado_path.write_text(
    json.dumps(_clean_json(resultado), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Archivo generado: {resultado_path}  ({len(resultado)} filas)")

# resumen_entidad (una fila por estado)
resumen_entidad_path = output_dir / "resumen_entidad.json"
resumen_entidad_path.write_text(
    json.dumps(_clean_json(resumen_entidad), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Archivo generado: {resumen_entidad_path}  ({len(resumen_entidad)} filas)")

Archivo generado: c:\Users\jose.valdez\Downloads\inform_new\reporte-new\public\resumen.json  (1272 filas)
Archivo generado: c:\Users\jose.valdez\Downloads\inform_new\reporte-new\public\resultado.json  (2058 filas)
Archivo generado: c:\Users\jose.valdez\Downloads\inform_new\reporte-new\public\resumen_entidad.json  (22 filas)


In [35]:
base_clues_total = (
    base["clues_imb"]
    .dropna()
    .astype(str)
    .str.strip()
)
base_clues_total = base_clues_total[base_clues_total != ""]

base_clues_unicas = base_clues_total.nunique()
base_clues_filas = len(base_clues_total)
base_entidades_esperadas = (
    base["entidad"]
    .dropna()
    .astype(str)
    .str.strip()
)
base_entidades_esperadas = base_entidades_esperadas[base_entidades_esperadas != ""].nunique()


## faltantes 

In [36]:
# TABLA - COMPLETITUD POR UNIDAD (todas las CLUES, sin filtrar)
# Renombrada a tu_faltantes para no sobreescribir tabla_unidades

n_preguntas = len(b)

consultorios = (
    base_an
    .groupby(["clues_imb","entidad",'nombre_de_la_unidad'],as_index=False)
    ["consultorio"]
    .max()
)
consultorios.rename(columns={"consultorio":"consultorios"}, inplace=True)

respondidas = (
    base_an
    .groupby(["clues_imb","entidad",'nombre_de_la_unidad'])
    .size()
    .reset_index(name="respondidas")
)

tu_faltantes = consultorios.merge(
    respondidas,
    on=["clues_imb","entidad",'nombre_de_la_unidad']
)

tu_faltantes["esperadas"] = tu_faltantes["consultorios"] * n_preguntas

tu_faltantes["porcentaje"] = (
    tu_faltantes["respondidas"] / tu_faltantes["esperadas"] * 100
).round(1).clip(upper=100)

tu_faltantes = (
    tu_faltantes
    .rename(columns={"clues_imb":"clues"})
    .sort_values("porcentaje")
)

In [37]:
# TABLA BASE por unidad (para la sección de faltantes)
# Renombrada a tu_base para no sobreescribir tabla_unidades

n_preguntas = len(b)

consultorios = (
    base_an
    .groupby(["clues_imb", "entidad", "nombre_de_la_unidad"], as_index=False)
    ["consultorio"]
    .max()
)
consultorios.rename(columns={"consultorio": "consultorios"}, inplace=True)

respondidas = (
    base_an
    .groupby(["clues_imb", "entidad", "nombre_de_la_unidad"])
    .size()
    .reset_index(name="respondidas")
)

tu_base = consultorios.merge(
    respondidas,
    on=["clues_imb", "entidad", "nombre_de_la_unidad"],
    how="left"
)

In [38]:
# ============================================
# TABLA 3 - PIVOT POR CONSULTORIO (SIN SUFIJOS)

# ============================================

# Solo respuestas del formulario
respuesta = base_an[base_an["tipo_registro"] == "respuesta"].copy()

# Limpieza de campos clave
respuesta["pregunta"] = respuesta["pregunta"].astype(str).str.strip()
respuesta["consultorio"] = pd.to_numeric(respuesta["consultorio"], errors="coerce")
respuesta = respuesta.dropna(subset=["pregunta", "consultorio"])
respuesta["consultorio"] = respuesta["consultorio"].astype(int)

# Elimina sufijos por consultorio (_1, _2, _3, ...)
respuesta["pregunta_base"] = respuesta["pregunta"].str.replace(r"_\d+$", "", regex=True)

# Pivot por unidad + consultorio usando solo pregunta base
pivot_consultorio = respuesta.pivot_table(
    index=["entidad", "clues_imb", "nombre_de_la_unidad", "consultorio"],
    columns="pregunta_base",
    values="valor",
    aggfunc="first"
).reset_index()

# Vista rápida
pivot_consultorio.head()

pregunta_base,entidad,clues_imb,nombre_de_la_unidad,consultorio,banco_de_altura_consultorio,banco_giratorio_consultorio,bascula_electronica__con_estadimetro_consultorio,bascula_electronica_con_estadimetro_consultorio,bascula_pesabebes_electronica_consultorio,bote_sanitario_con_pedal_consultorio,...,silla_para_el_paciente_consultorio,sillon_giratorio_de_respaldo_bajo_consultorio,termo_congelante_integrado_para_transporte_y_conservacion_de_vacunas_consultorio,termometro_de_vastago_consultorio,termometro_digital_consultorio,tijera_recta_consultorio,torundera_con_tapa_de_acero_inoxidable_consultorio,turno_consultorio,vaso_contenedor__de_vacunas_consultorio,vaso_contenedor_de_vacunas_consultorio
0,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,1,1,0,0,NaN,0,0,...,1,1,NaN,NaN,NaN,NaN,NaN,Matutino,NaN,NaN
1,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,2,1,1,1,NaN,0,1,...,1,1,NaN,NaN,NaN,NaN,NaN,Matutino,NaN,NaN
2,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Matutino,NaN,NaN
3,BAJA CALIFORNIA,BCIMB000302,HÉROES DE LA INDEPENDENCIA,1,0,1,0,NaN,0,0,...,1,1,0,0,0,1,1,Matutino,0,NaN
4,BAJA CALIFORNIA,BCIMB000314,VALLE DE LA TRINIDAD,1,1,1,1,NaN,1,1,...,1,1,0,0,0,0,1,Matutino,0,NaN


In [39]:
# Conteo de preguntas unicas pivoteadas
n_preguntas_pivoteadas = respuesta["pregunta_base"].nunique()
print("Preguntas unicas pivoteadas:", n_preguntas_pivoteadas)

# Validacion contra columnas del pivot (excluyendo llaves del indice)
llaves = ["entidad", "clues_imb", "nombre_de_la_unidad", "consultorio"]
print("Columnas de preguntas en pivot:", len([c for c in pivot_consultorio.columns if c not in llaves]))

Preguntas unicas pivoteadas: 49
Columnas de preguntas en pivot: 49


In [40]:
# ============================================
# TABLA 4 - LLENADO POR CLUES Y CONSULTORIO
# Base esperada: preguntas de b (excluyendo 'turno' y filas 0,1,2,5)
# ============================================

import re
import unicodedata


def normalizar_pregunta(x: str) -> str:
    s = str(x).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"_\d+$", "", s)                # quita sufijo numerico
    s = re.sub(r"[^a-z0-9]+", "_", s)           # espacios/simbolos -> _
    s = re.sub(r"_+", "_", s).strip("_")       # limpia _ repetidos
    s = re.sub(r"_consultorio$", "", s)         # quita sufijo fijo consultorio
    return s

# 1) Catalogo esperado desde b (quitando filas especificas)
b_filtrado = b.drop(index=[0, 1, 2, 5], errors="ignore").copy()

catalogo_b = b_filtrado["preguntas"].dropna().astype(str).str.strip()
catalogo_b = catalogo_b[catalogo_b != ""]
catalogo_b_norm = catalogo_b.map(normalizar_pregunta)
catalogo_b_norm = catalogo_b_norm[catalogo_b_norm.str.contains("turno") == False]
set_catalogo_b = set(catalogo_b_norm.tolist())
n_preguntas_esperadas = len(set_catalogo_b)

print(f"Preguntas esperadas desde b filtrado (sin turno): {n_preguntas_esperadas}")

# 2) Respuestas del formulario por consultorio
resp = base_an[base_an["tipo_registro"] == "respuesta"].copy()
resp["pregunta"] = resp["pregunta"].astype(str).str.strip()
resp["pregunta_norm"] = resp["pregunta"].map(normalizar_pregunta)
resp["consultorio"] = pd.to_numeric(resp["consultorio"], errors="coerce")
resp = resp.dropna(subset=["clues_imb", "consultorio", "pregunta_norm"])
resp = resp[resp["pregunta_norm"] != ""]
resp = resp[resp["pregunta_norm"].str.contains("turno") == False]
resp["consultorio"] = resp["consultorio"].astype(int)

keys = ["entidad", "clues_imb", "nombre_de_la_unidad", "consultorio"]

# 3) Calcula por consultorio: respondidas, faltantes, extras y porcentaje
filas = []
faltantes_detalle = []

for k, grp in resp.groupby(keys, dropna=False):
    preguntas_unicas = set(grp["pregunta_norm"].dropna().astype(str).str.strip())
    preguntas_unicas.discard("")

    respondidas_validas = len(preguntas_unicas & set_catalogo_b)
    faltantes_set = set_catalogo_b - preguntas_unicas
    extras_set = preguntas_unicas - set_catalogo_b

    faltantes = sorted(faltantes_set)
    extras = sorted(extras_set)

    porcentaje = round((respondidas_validas / n_preguntas_esperadas) * 100, 1) if n_preguntas_esperadas else 0.0

    filas.append({
        "entidad": k[0],
        "clues_imb": k[1],
        "nombre_de_la_unidad": k[2],
        "consultorio": k[3],
        "preguntas_esperadas": n_preguntas_esperadas,
        "preguntas_respondidas_validas": respondidas_validas,
        "faltantes": len(faltantes),
        "extras_fuera_catalogo": len(extras),
        "porcentaje_llenado": porcentaje,
        "lista_faltantes": ", ".join(faltantes),
        "lista_extras": ", ".join(extras),
    })

    for pregunta_faltante in faltantes:
        faltantes_detalle.append({
            "entidad": k[0],
            "clues_imb": k[1],
            "nombre_de_la_unidad": k[2],
            "consultorio": k[3],
            "pregunta_faltante": pregunta_faltante,
        })

tabla_llenado_consultorio = pd.DataFrame(filas).sort_values(
    ["clues_imb", "consultorio"],
    ascending=[True, True]
).reset_index(drop=True)

# 4) Detalle de faltantes por consultorio
faltantes_por_consultorio = pd.DataFrame(faltantes_detalle).sort_values(
    ["clues_imb", "consultorio", "pregunta_faltante"],
    ascending=[True, True, True]
).reset_index(drop=True)

# 5) Resumen por CLUES (acumulado de consultorios)
resumen_llenado_clues = (
    tabla_llenado_consultorio
    .groupby(["entidad", "clues_imb", "nombre_de_la_unidad"], as_index=False)
    .agg(
        consultorios_reportados=("consultorio", "nunique"),
        preguntas_esperadas_por_consultorio=("preguntas_esperadas", "max"),
        preguntas_respondidas_validas_total=("preguntas_respondidas_validas", "sum"),
        faltantes_total=("faltantes", "sum"),
        extras_fuera_catalogo_total=("extras_fuera_catalogo", "sum"),
    )
)

resumen_llenado_clues["preguntas_esperadas_total"] = (
    resumen_llenado_clues["consultorios_reportados"]
    * resumen_llenado_clues["preguntas_esperadas_por_consultorio"]
)

resumen_llenado_clues["porcentaje_llenado_total"] = (
    resumen_llenado_clues["preguntas_respondidas_validas_total"]
    / resumen_llenado_clues["preguntas_esperadas_total"]
    * 100
).round(1)

print("\nVista por consultorio (primeras filas):")
display(tabla_llenado_consultorio.head())

print("\nDetalle de faltantes con consultorio (primeras filas):")
display(faltantes_por_consultorio.head())

Preguntas esperadas desde b filtrado (sin turno): 44

Vista por consultorio (primeras filas):


,entidad,clues_imb,nombre_de_la_unidad,consultorio,preguntas_esperadas,preguntas_respondidas_validas,faltantes,extras_fuera_catalogo,porcentaje_llenado,lista_faltantes,lista_extras
0,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,1,44,30,14,0,68.2,"cinta_metrica, estetoscopio_capsula_doble, mar...",
1,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,2,44,30,14,0,68.2,"estadimetro_pediatrico, estetoscopio_capsula_d...",
2,BAJA CALIFORNIA,BCIMB000302,HÉROES DE LA INDEPENDENCIA,1,44,44,0,0,100.0,,
3,BAJA CALIFORNIA,BCIMB000314,VALLE DE LA TRINIDAD,1,44,44,0,0,100.0,,
4,BAJA CALIFORNIA,BCIMB000466,ORIZABA,1,44,2,42,0,4.5,"banco_giratorio, bascula_electronica_con_estad...",



Detalle de faltantes con consultorio (primeras filas):


,entidad,clues_imb,nombre_de_la_unidad,consultorio,pregunta_faltante
0,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,1,cinta_metrica
1,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,1,estetoscopio_capsula_doble
2,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,1,martillo_percusor
3,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,1,mesa_de_mayo_de_acero_inoxidable
4,BAJA CALIFORNIA,BCIMB000092,FRACCIONAMIENTO MAR,1,mesa_pasteur


In [41]:
# Consultorios con >= 90% de llenado
tabla_llenado_90 = tabla_llenado_consultorio[
    tabla_llenado_consultorio["porcentaje_llenado"] >= 80
].reset_index(drop=True)

print(f"Consultorios con >= 90% de llenado: {len(tabla_llenado_90)}")

# Faltantes solo de esos consultorios
clues_cons_90 = tabla_llenado_90[["clues_imb", "consultorio"]]

faltantes_90 = faltantes_por_consultorio.merge(
    clues_cons_90,
    on=["clues_imb", "consultorio"],
    how="inner"
).reset_index(drop=True)

print(f"Registros de faltantes en consultorios >= 90%: {len(faltantes_90)}")

display(tabla_llenado_90)
display(faltantes_90)

Consultorios con >= 90% de llenado: 2059
Registros de faltantes en consultorios >= 90%: 2182


,entidad,clues_imb,nombre_de_la_unidad,consultorio,preguntas_esperadas,preguntas_respondidas_validas,faltantes,extras_fuera_catalogo,porcentaje_llenado,lista_faltantes,lista_extras
0,BAJA CALIFORNIA,BCIMB000302,HÉROES DE LA INDEPENDENCIA,1,44,44,0,0,100.0,,
1,BAJA CALIFORNIA,BCIMB000314,VALLE DE LA TRINIDAD,1,44,44,0,0,100.0,,
2,BAJA CALIFORNIA,BCIMB000536,DURANGO,1,44,44,0,0,100.0,,
3,BAJA CALIFORNIA,BCIMB000990,PEDREGAL DE SANTA JULIA,1,44,44,0,0,100.0,,
4,BAJA CALIFORNIA SUR,BSIMB000013,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,1,44,41,3,0,93.2,"charola_de_mayo_de_acero_inoxidable, contenedo...",
...,...,...,...,...,...,...,...,...,...,...,...
2054,ZACATECAS,ZSIMB002435,CENTRO DE SALUD LA PINTA,2,44,44,0,0,100.0,,
2055,ZACATECAS,ZSIMB002522,CENTRO DE SALUD FRESNILLO 2,1,44,44,0,0,100.0,,
2056,ZACATECAS,ZSIMB002522,CENTRO DE SALUD FRESNILLO 2,2,44,44,0,0,100.0,,
2057,ZACATECAS,ZSIMB002522,CENTRO DE SALUD FRESNILLO 2,3,44,44,0,0,100.0,,


,entidad,clues_imb,nombre_de_la_unidad,consultorio,pregunta_faltante
0,BAJA CALIFORNIA SUR,BSIMB000013,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,1,charola_de_mayo_de_acero_inoxidable
1,BAJA CALIFORNIA SUR,BSIMB000013,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,1,contenedor_rigido_para_rpbi
2,BAJA CALIFORNIA SUR,BSIMB000013,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,1,escritorio_medico
3,BAJA CALIFORNIA SUR,BSIMB000013,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,3,banco_giratorio
4,BAJA CALIFORNIA SUR,BSIMB000030,CENTRO DE SALUD BENITO JUÁREZ,1,bote_sanitario_con_pedal
...,...,...,...,...,...
2177,ZACATECAS,ZSIMB000434,CENTRO DE SALUD COL. TIERRA Y LIBERTAD,1,estetoscopio_pinard_o_doppler_fetal_portatil
2178,ZACATECAS,ZSIMB001402,CENTRO DE SALUD SANTA LUCÍA DE LA SIERRA,1,banco_giratorio
2179,ZACATECAS,ZSIMB001641,CENTRO DE SALUD ZACATECAS,2,estadimetro_pediatrico
2180,ZACATECAS,ZSIMB001641,CENTRO DE SALUD ZACATECAS,4,banco_giratorio


In [42]:
from pathlib import Path
from datetime import date

ruta = Path(fr"C:\Users\{usuario}\Downloads\faltantes_90_{date.today()}.xlsx")
faltantes_90.to_excel(ruta, index=False)
print(f"Archivo guardado en: {ruta}")

Archivo guardado en: C:\Users\jose.valdez\Downloads\faltantes_90_2026-08-21.xlsx


In [43]:
faltantes_90_agrupado = (
    faltantes_90
    .groupby(["entidad", "clues_imb", "nombre_de_la_unidad", "consultorio"], as_index=False)
    .agg(
        n_faltantes=("pregunta_faltante", "count"),
        preguntas_faltantes=("pregunta_faltante", lambda x: ", ".join(sorted(x))),
    )
    .sort_values(["clues_imb", "consultorio"])
    .reset_index(drop=True)
)

print(f"Filas agrupadas: {len(faltantes_90_agrupado)}")
faltantes_90_agrupado

Filas agrupadas: 709


,entidad,clues_imb,nombre_de_la_unidad,consultorio,n_faltantes,preguntas_faltantes
0,BAJA CALIFORNIA SUR,BSIMB000013,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,1,3,"charola_de_mayo_de_acero_inoxidable, contenedo..."
1,BAJA CALIFORNIA SUR,BSIMB000013,CENTRO DE SALUD CIUDAD CONSTITUCIÓN,3,1,banco_giratorio
2,BAJA CALIFORNIA SUR,BSIMB000030,CENTRO DE SALUD BENITO JUÁREZ,1,2,"bote_sanitario_con_pedal, caja_portalaminilla_..."
3,BAJA CALIFORNIA SUR,BSIMB000071,CENTRO DE SALUD PUERTO SAN CARLOS,1,1,equipo_de_computo
4,BAJA CALIFORNIA SUR,BSIMB000100,CENTRO DE SALUD VILLA MORELOS,1,5,"banco_de_altura, banco_giratorio, estadimetro_..."
...,...,...,...,...,...,...
704,ZACATECAS,ZSIMB000434,CENTRO DE SALUD COL. TIERRA Y LIBERTAD,1,2,"carta_snellen_con_marco, estetoscopio_pinard_o..."
705,ZACATECAS,ZSIMB001402,CENTRO DE SALUD SANTA LUCÍA DE LA SIERRA,1,1,banco_giratorio
706,ZACATECAS,ZSIMB001641,CENTRO DE SALUD ZACATECAS,2,1,estadimetro_pediatrico
707,ZACATECAS,ZSIMB001641,CENTRO DE SALUD ZACATECAS,4,1,banco_giratorio


In [44]:
# Exportar tabla de faltantes agrupados a public/faltantes.json
faltantes_path = output_dir / "faltantes.json"
faltantes_path.write_text(
    json.dumps(_clean_json(faltantes_90_agrupado), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Archivo generado: {faltantes_path}  ({len(faltantes_90_agrupado)} filas)")

Archivo generado: c:\Users\jose.valdez\Downloads\inform_new\reporte-new\public\faltantes.json  (709 filas)


In [45]:
# Unidades con MAYOR cobertura por entidad (top 1 por estado)
# Usando resumen_llenado_clues que tiene porcentaje_llenado_total
top_por_entidad = (
    resumen_llenado_clues
    .sort_values("porcentaje_llenado_total", ascending=False)
    .groupby("entidad", as_index=False)
    .first()[["entidad", "clues_imb", "nombre_de_la_unidad", "consultorios_reportados", "porcentaje_llenado_total"]]
    .sort_values("entidad")
    .reset_index(drop=True)
)

print(f"Entidades representadas: {len(top_por_entidad)}")
display(top_por_entidad.style.background_gradient(
    subset=["porcentaje_llenado_total"], cmap="RdYlGn", vmin=90, vmax=100
))

Entidades representadas: 22


,entidad,clues_imb,nombre_de_la_unidad,consultorios_reportados,porcentaje_llenado_total
0,BAJA CALIFORNIA,BCIMB000302,HÉROES DE LA INDEPENDENCIA,1,100.000000
1,BAJA CALIFORNIA SUR,BSIMB000532,CENTRO DE SALUD SAN MIGUEL DE COMONDÚ,1,100.000000
2,CAMPECHE,CCIMB000213,CENTRO AVANZADO DE ATENCIÓN PRIMARIA A LA SALUD SAN ANTONIO CÁRDENAS,1,100.000000
3,CHIAPAS,CSIMB000501,NUEVA CONCORDIA,1,100.000000
4,CIUDAD DE MEXICO,DFIMB001332,C.S.T-III AMPLIACIÓN PRESIDENTES,1,100.000000
5,COLIMA,CMIMB000383,CENTRO DE SALUD LA SIDRA,1,100.000000
6,GUERRERO,GRIMB000275,U-03 COL. ZAPATA II,1,100.000000
7,HIDALGO,HGIMB002391,ABRAHAM KANAN HUEBE,2,100.000000
8,MEXICO,MCIMB000733,SAN MARTÍN,1,100.000000
9,MICHOACAN DE OCAMPO,MNIMB002914,CENTRO DE SALUD COL. ERÉNDIRA,1,100.000000


In [46]:
# Todas las unidades de San Luis Potosí
slp = (
    resumen_llenado_clues[resumen_llenado_clues["entidad"].str.upper().str.contains("SAN LUIS")]
    [["clues_imb", "nombre_de_la_unidad", "consultorios_reportados",
      "preguntas_respondidas_validas_total", "preguntas_esperadas_total",
      "faltantes_total", "porcentaje_llenado_total"]]
    .sort_values("porcentaje_llenado_total", ascending=False)
    .reset_index(drop=True)
)

print(f"Unidades SLP con >=90%: {len(slp)}")
print(f"Promedio llenado: {slp['porcentaje_llenado_total'].mean():.1f}%")
print(f"Consultorios reportados total: {slp['consultorios_reportados'].sum()}")
display(slp)

Unidades SLP con >=90%: 27
Promedio llenado: 94.4%
Consultorios reportados total: 64


,clues_imb,nombre_de_la_unidad,consultorios_reportados,preguntas_respondidas_validas_total,preguntas_esperadas_total,faltantes_total,porcentaje_llenado_total
0,SPIMB000252,PIMIENTA,1,44,44,0,100.0
1,SPIMB000264,COL. JUÁREZ,2,88,88,0,100.0
2,SPIMB000696,SAUCITO,1,44,44,0,100.0
3,SPIMB001331,NUEVO TAMPAON,1,44,44,0,100.0
4,SPIMB000520,CENTRO DE SALUD LOS POCITOS,1,44,44,0,100.0
5,SPIMB000993,PONCIANO ARRIAGA,1,44,44,0,100.0
6,SPIMB000981,CENTRO DE SALUD 1RO. DE MAYO,2,88,88,0,100.0
7,SPIMB002340,CENTRO DE SALUD FRANCISCO VILLA,1,44,44,0,100.0
8,SPIMB002195,CENTRO DE SALUD 6 DE JUNIO,2,88,88,0,100.0
9,SPIMB002101,16 DE SEPTIEMBRE,2,88,88,0,100.0


In [47]:
tabla_unidades

,clues,entidad,nombre_de_la_unidad,consultorios,respondidas,esperadas,porcentaje
763,MCIMB001713,MEXICO,CEAPS SAN MIGUEL CHAPULTEPEC BICENTENARIO,1.0,2,48.0,100.0
895,MCIMB007243,MEXICO,VALLE DE BRAVO,2.0,47,96.0,100.0
858,MCIMB006625,MEXICO,CENTRO DE SALUD CARACOLES,2.0,47,96.0,100.0
819,MCIMB003912,MEXICO,ESTADO DE MÉXICO,6.0,142,288.0,100.0
810,MCIMB003796,MEXICO,SAN FRANCISCO CUAUTLALPAN,6.0,143,288.0,100.0
...,...,...,...,...,...,...,...
1151,QRIMB000182,QUINTANA ROO,CENTRO DE SALUD RURAL SANTA ROSA SEGUNDO,1.0,60,48.0,100.0
1153,QRIMB000252,QUINTANA ROO,CENTRO DE SALUD RURAL X-YATIL,2.0,98,96.0,100.0
1282,SLIMB001472,SINALOA,LA NORIA DE SAN ANTONIO (LA NORIA),1.0,48,48.0,100.0
1191,QRIMB000795,QUINTANA ROO,CENTRO DE SALUD RURAL DOS AGUADAS,1.0,48,48.0,100.0
